# Consultar la tabla limpia

In [20]:
import sys
from pathlib import Path
import pandas as pd
raiz = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data/interim").exists())
sys.path.insert(0, str(raiz / "ai/src/data_prep"))

import pandas as pd
from ccpc import leer_tabla
pd.set_option("display.max_columns", None)

In [21]:
df = leer_tabla(anios=[2024])
df.head()

,AELEC,FELECCION,EDOCVE,EDONOM,MPIOCVE,MPIONOM,SECCION,TIPOSEC,DEF,DEL,SEXO,EDAD,LN,SV,NV,NS,fecha_eleccion,tipo_eleccion,grupo_edad,flag_regla_ln,flag_negativo,flag_duplicado,flag_codigo,flag_edad,flag_revision
0,2024,06/02/2024,1,AGUASCALIENTES,1,Aguascalientes,1,U,3,6,0,18,15,13,2,0,2024-06-02,presidencial,18 a 24,False,False,False,False,False,False
1,2024,06/02/2024,1,AGUASCALIENTES,1,Aguascalientes,1,U,3,6,0,19,23,17,6,0,2024-06-02,presidencial,18 a 24,False,False,False,False,False,False
2,2024,06/02/2024,1,AGUASCALIENTES,1,Aguascalientes,1,U,3,6,0,20,20,13,7,0,2024-06-02,presidencial,18 a 24,False,False,False,False,False,False
3,2024,06/02/2024,1,AGUASCALIENTES,1,Aguascalientes,1,U,3,6,0,21,24,16,8,0,2024-06-02,presidencial,18 a 24,False,False,False,False,False,False
4,2024,06/02/2024,1,AGUASCALIENTES,1,Aguascalientes,1,U,3,6,0,22,21,13,8,0,2024-06-02,presidencial,18 a 24,False,False,False,False,False,False


In [22]:
# Solo algunas columnas, de varias elecciones
df = leer_tabla(anios=[2018, 2021], columnas=["AELEC", "EDONOM", "MPIONOM", "SECCION", "LN", "SV", "NV"])
df.head()

,AELEC,EDONOM,MPIONOM,SECCION,LN,SV,NV
0,2018,AGUASCALIENTES,Aguascalientes,1,24,18,6
1,2018,AGUASCALIENTES,Aguascalientes,1,23,20,3
2,2018,AGUASCALIENTES,Aguascalientes,1,23,19,4
3,2018,AGUASCALIENTES,Aguascalientes,1,19,12,7
4,2018,AGUASCALIENTES,Aguascalientes,1,19,10,9


In [23]:
sys.path.insert(0, str(raiz / "scripts"))
from ccpc import ANIOS
from features_por_municipio import SALIDA as CARPETA_FEATURES


def leer_features(anios=None, columnas=None) -> pd.DataFrame:
    if anios is None:
        anios = ANIOS
    elif isinstance(anios, int):
        anios = [anios]
    archivos = {a: CARPETA_FEATURES / f"features_{a}.parquet" for a in anios}
    faltan = [a for a, archivo in archivos.items() if not archivo.exists()]
    if faltan:
        raise FileNotFoundError(f"Faltan las features de {faltan}. Corre: python scripts/features_por_municipio.py")
    piezas = [pd.read_parquet(archivo, columns=columnas) for archivo in archivos.values()]
    return pd.concat(piezas, ignore_index=True)

In [ ]:
feats = leer_features(anios=[2024])
feats[(feats["MPIONOM"] == "Puebla") & (feats["EDONOM"] == "PUEBLA")]


,AELEC,EDOCVE,MPIOCVE,EDONOM,MPIONOM,TOTAL_SECCIONES,SEXO_mean,SEXO_std,SEXO_max,SEXO_min,EDAD_mean,EDAD_std,EDAD_max,EDAD_min,LN_mean,LN_std,LN_max,LN_min,SV_mean,SV_std,SV_max,SV_min,NV_mean,NV_std,NV_max,NV_min,NS_mean,NS_std,NS_max,NS_min
1699,2024,21,115,PUEBLA,Puebla,817,0.512401,0.499848,1,0,53.604352,21.361903,113,18,11.890991,15.38982,283,1,6.693575,7.79035,124,0,4.445071,7.895295,180,0,0.752346,2.511219,53,0


In [68]:
feats_2 = leer_features(anios=[2018])
feats_2[(feats_2["MPIONOM"] == "Puebla") & (feats_2["EDONOM"] == "PUEBLA")]

,AELEC,EDOCVE,MPIOCVE,EDONOM,MPIONOM,TOTAL_SECCIONES,SEXO_mean,SEXO_std,SEXO_max,SEXO_min,EDAD_mean,EDAD_std,EDAD_max,EDAD_min,LN_mean,LN_std,LN_max,LN_min,SV_mean,SV_std,SV_max,SV_min,NV_mean,NV_std,NV_max,NV_min,NS_mean,NS_std,NS_max,NS_min
1687,2018,21,115,PUEBLA,Puebla,768,0.51171,0.499865,1,0,53.451208,21.368413,113,18,11.900539,17.163329,246,1,6.897147,9.409363,164,0,4.141645,7.487642,147,0,0.861747,3.566211,96,0


In [ ]:

feats = leer_features(columnas=["AELEC", "EDONOM", "MPIONOM", "TOTAL_SECCIONES", "LN_mean", "NV_mean"])
feats.groupby("AELEC").size()

AELEC
2009    2458
2012    2455
2015    2458
2018    2463
2021    2468
2024    2477
dtype: int64